# Туториал: S3 Bridge streaming (`streaming.mode=s3_bridge`)

Цель: producer публикует final chunks в S3, consumer скачивает и отдаёт `SharedSample` в обучение.

Подходит для:
- раздельных машин/окружений
- decoupled producer-consumer topologies


In [ ]:
from __future__ import annotations

import sys
import platform
from pathlib import Path

import torch
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset

def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        return compose(config_name=cfg_path.stem, overrides=overrides or [])

print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')


## Подготовка

Перед запуском проверьте:
1. `streaming.s3.bucket` заполнен
2. права на `PutObject/GetObject/DeleteObject`
3. при S3-compatible storage указан `streaming.s3.endpoint_url`

По умолчанию удаление remote chunk: `delete_remote_after=consume`.


In [ ]:
S3_BUCKET = ""  # <- заполните
RUN_DEMO = False   # поставьте True когда готовы запускать

overrides = [
    "streaming=s3_bridge_streaming",
    "data.path=./data",
    "train.device=cuda:0" if torch.cuda.is_available() else "train.device=cpu",
    "collector.device=cuda:1" if torch.cuda.device_count() >= 2 else "collector.device=null",
    "collector.mode=auto",
    "data.enabled_datasets=[coco2017,cc12m]",
    "data.dataset_overrides.coco2017.models=[clip_vit_b32]",
    "data.dataset_overrides.cc12m.models=[clip_vit_b32]",
    f"streaming.s3.bucket={S3_BUCKET}",
    "streaming.s3.prefix=diffusion-meta-opt/streaming",
    "streaming.s3.max_remote_chunks=100",
    "streaming.consumer.delete_remote_after=consume",
]

cfg = load_hydra_cfg(overrides=overrides)
print('streaming mode =', cfg.streaming.mode)
print(OmegaConf.to_yaml(cfg.streaming, resolve=True))


In [ ]:
if not RUN_DEMO:
    print('RUN_DEMO=False -> только preview конфигурации. Для запуска поставьте RUN_DEMO=True.')
elif not str(S3_BUCKET).strip():
    raise ValueError('Заполните S3_BUCKET перед запуском demo')
else:
    collector = CollectorService(cfg)
    dataset = SharedModelDataset(collector)
    collector.start()

    consumed = 0
    try:
        for step in range(100):
            if not collector.is_async_mode:
                dataset.maybe_collect(step)

            sample = dataset.try_next_sample()
            if sample is None:
                continue
            consumed += 1

        print('consumed:', consumed)
        print('remote ready metric:', dataset.cache_size())
        print('collector stats:', collector.stats())
    finally:
        dataset.close()
        collector.shutdown()


## Быстрая диагностика

- Нет chunk-ов в S3: проверьте bucket/prefix/credentials.
- 403/AccessDenied: не хватает IAM прав.
- Потребитель не читает: проверьте `delete_remote_after` и `consumer.local_cache_dir`.
